# Colab: ImageNet SSL 预训练（ViT-Tiny）

这个 notebook 按当前仓库的 `train_ssl_freq_mae.py` 写好了 Colab 流程，目标是：

1. 克隆 `https://github.com/BoldHu/test.git`
2. 准备 ImageNet 数据，并放到仓库能直接识别的位置
3. 先做一个小型 smoke test
4. smoke test 通过后，再正式启动 `train_ssl_freq_mae.py` 的 ViT-Tiny 预训练

默认策略：

- 数据尽量放在 `/content`，这是 Colab 里最快的读取路径
- 日志和 checkpoint 放到 Google Drive，避免断连丢失
- 如果你担心 230G 本地磁盘不够，可以把 `USE_DRIVE_FOR_DATA` 改成 `True`
- Kaggle 路径优先兼容 `ILSVRC/Data/CLS-LOC/{train,val}`，也兼容 `train/val`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/BoldHu/test.git'
REPO_DIR = Path('/content/test')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'timm>=1.0.0',
        'einops>=0.7.0',
        'tensorboard>=2.16.0',
        'matplotlib>=3.7.0',
        'tqdm>=4.65.0',
        'kaggle>=1.6.0',
    ],
    check=True,
)

print('Repo ready:', REPO_DIR)


In [ ]:
from pathlib import Path
import shutil
import torch

KAGGLE_COMPETITION = 'imagenet-object-localization-challenge'

# 最快方案：False，把数据放到 /content；如果本地空间紧张，再改成 True。
USE_DRIVE_FOR_DATA = False

# 如果 DATA_ROOT 里还没有 ImageNet，会自动尝试从 Kaggle 顺序下载并解压。
FORCE_DOWNLOAD_FROM_KAGGLE = True

LOCAL_DATA_ROOT = Path('/content/imagenet')
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/datasets/imagenet')
DATA_ROOT = DRIVE_DATA_ROOT if USE_DRIVE_FOR_DATA else LOCAL_DATA_ROOT
RUNS_ROOT = Path('/content/drive/MyDrive/imagenet_ssl_runs')
SMOKE_ROOT = Path('/content/imagenet_smoke')

TIMM_TINY_MODEL = 'vit_tiny_patch16_224.augreg_in21k_ft_in1k'
MASK_RATIO = 0.75
FULL_EXPERIMENT = 'vit_tiny_ssl_imagenet_colab'
SMOKE_EXPERIMENT = 'vit_tiny_ssl_smoke'
FULL_EPOCHS = 800

# 当前仓库默认是 base + MAE decoder。这里显式把 decoder 也缩小，形成更合理的 tiny 版本。
DECODER_EMB_DIM = 192
DECODER_LAYERS = 4
DECODER_HEADS = 3

if not torch.cuda.is_available():
    raise RuntimeError('请在 Colab 里切到 GPU Runtime 后再继续。')

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_mem_gb >= 35:
    TRAIN_BATCH_SIZE = 128
    VAL_BATCH_SIZE = 128
    GRAD_ACCUM_STEPS = 2
    NUM_WORKERS = 4
elif gpu_mem_gb >= 20:
    TRAIN_BATCH_SIZE = 64
    VAL_BATCH_SIZE = 64
    GRAD_ACCUM_STEPS = 4
    NUM_WORKERS = 4
else:
    TRAIN_BATCH_SIZE = 32
    VAL_BATCH_SIZE = 32
    GRAD_ACCUM_STEPS = 8
    NUM_WORKERS = 2

effective_bs = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS

RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def print_disk(path: Path):
    usage = shutil.disk_usage(path)
    print(f'{path}: free={usage.free / 1024**3:.1f} GiB | used={usage.used / 1024**3:.1f} GiB | total={usage.total / 1024**3:.1f} GiB')

print('GPU:', gpu_name)
print(f'GPU memory: {gpu_mem_gb:.1f} GiB')
print('DATA_ROOT:', DATA_ROOT)
print('RUNS_ROOT:', RUNS_ROOT)
print('TIMM_TINY_MODEL:', TIMM_TINY_MODEL)
print('Suggested batch size:', TRAIN_BATCH_SIZE)
print('Suggested grad_accum_steps:', GRAD_ACCUM_STEPS)
print('Effective batch size:', effective_bs)
print_disk(Path('/content'))
print_disk(Path('/content/drive'))


## Kaggle 认证

如果你已经把 ImageNet 放进了 `DATA_ROOT`，这一格可以跳过。

如果要从 Kaggle 下载：

- 先在 Kaggle 网页端接受 `ImageNet Object Localization Challenge` 的使用条款
- 准备好 `kaggle.json`
- 执行下一格把 `kaggle.json` 上传到 Colab


In [ ]:
from google.colab import files
from pathlib import Path
import os
import subprocess

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('没有检测到 kaggle.json，请重新上传。')
    kaggle_json.write_bytes(uploaded['kaggle.json'])

os.chmod(kaggle_json, 0o600)
subprocess.run(['kaggle', 'competitions', 'files', '-c', KAGGLE_COMPETITION], check=True)
print('Kaggle auth is ready.')


In [ ]:
from pathlib import Path
import os
import re
import shlex
import shutil
import subprocess
import sys

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG'}

def find_images(root: Path, limit: int | None = None):
    images = []
    for path in sorted(root.rglob('*')):
        if path.is_file() and path.suffix in IMG_EXTS:
            images.append(path)
            if limit is not None and len(images) >= limit:
                break
    return images

def resolve_imagenet_layout(data_root: Path):
    candidates = [
        (data_root / 'train', data_root / 'val'),
        (data_root / 'ILSVRC' / 'Data' / 'CLS-LOC' / 'train', data_root / 'ILSVRC' / 'Data' / 'CLS-LOC' / 'val'),
        (data_root / 'ILSVRC' / 'Data' / 'train', data_root / 'ILSVRC' / 'Data' / 'val'),
    ]
    for train_dir, val_dir in candidates:
        if train_dir.exists() and val_dir.exists():
            return train_dir, val_dir
    return None, None

def download_and_extract_kaggle_imagenet(data_root: Path, competition: str, stage_dir: Path):
    from kaggle.api.kaggle_api_extended import KaggleApi

    data_root.mkdir(parents=True, exist_ok=True)
    stage_dir.mkdir(parents=True, exist_ok=True)

    api = KaggleApi()
    api.authenticate()
    files = api.competition_list_files(competition)

    archive_names = []
    for item in files:
        name = item.name
        lower = name.lower()
        if lower.endswith(('.csv', '.txt', '.json', '.xml')):
            continue
        if any(token in lower for token in ('train', 'val', 'valid', 'ilsvrc', 'cls-loc')):
            archive_names.append(name)

    archive_names = sorted(set(archive_names))
    if not archive_names:
        raise RuntimeError('Kaggle 列表里没有找到可下载的 ImageNet 档案文件。')

    print('Archives to download/extract:')
    for name in archive_names:
        print(' -', name)

    for name in archive_names:
        if re.search(r'\\.zip\\.\\d+$', name):
            raise RuntimeError(f'检测到分卷压缩文件 {name}，当前 notebook 没有自动拼接这类分卷，请改用已准备好的 Drive 数据副本。')

        archive_path = stage_dir / name
        if not archive_path.exists():
            print(f'Downloading {name} ...')
            api.competition_download_file(competition, name, path=str(stage_dir), force=False, quiet=False)

        print(f'Extracting {archive_path.name} -> {data_root}')
        if archive_path.suffix == '.zip':
            subprocess.run(['unzip', '-qo', str(archive_path), '-d', str(data_root)], check=True)
        else:
            shutil.unpack_archive(str(archive_path), str(data_root))

        archive_path.unlink(missing_ok=True)
        print_disk(data_root)

def ensure_repo_data_link(repo_dir: Path, data_root: Path):
    data_dir = repo_dir / 'data'
    data_dir.mkdir(parents=True, exist_ok=True)
    link_path = data_dir / 'imagenet'
    if link_path.is_symlink() or link_path.exists():
        return link_path
    os.symlink(data_root, link_path, target_is_directory=True)
    return link_path

train_dir, val_dir = resolve_imagenet_layout(DATA_ROOT)

if (train_dir is None or val_dir is None) and FORCE_DOWNLOAD_FROM_KAGGLE:
    stage_dir = Path('/content/kaggle_stage')
    download_and_extract_kaggle_imagenet(DATA_ROOT, KAGGLE_COMPETITION, stage_dir)
    train_dir, val_dir = resolve_imagenet_layout(DATA_ROOT)

if train_dir is None or val_dir is None:
    raise FileNotFoundError(
        f'没有在 {DATA_ROOT} 下找到可用的 ImageNet 目录。'\
        "\n如果你已经有数据，请确认是 train/val 或 ILSVRC/Data/CLS-LOC/train,val 结构。"\
        "\n如果你要在线下载，请把 FORCE_DOWNLOAD_FROM_KAGGLE 改成 True 再执行这一格。"
    )

repo_data_link = ensure_repo_data_link(REPO_DIR, DATA_ROOT)

print('Resolved train_dir:', train_dir)
print('Resolved val_dir:', val_dir)
print('Repo data link:', repo_data_link)
print('Train image sample count:', len(find_images(train_dir, limit=8)))
print('Val image sample count:', len(find_images(val_dir, limit=8)))


In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess
import sys

def build_smoke_subset(train_src: Path, val_src: Path, smoke_root: Path, train_count: int = 64, val_count: int = 16):
    if smoke_root.exists():
        shutil.rmtree(smoke_root)

    for split, src, count in [('train', train_src, train_count), ('val', val_src, val_count)]:
        dst_dir = smoke_root / split / 'dummy'
        dst_dir.mkdir(parents=True, exist_ok=True)
        for idx, img_path in enumerate(find_images(src, limit=count)):
            dst_path = dst_dir / f'{idx:05d}{img_path.suffix.lower()}'
            os.symlink(img_path, dst_path)

    return smoke_root / 'train', smoke_root / 'val'

def find_resume_ckpt(output_root: Path, experiment: str):
    ckpt_dir = output_root / experiment / 'checkpoints'
    for name in ('last_checkpoint_norm_pix.pth', 'last_checkpoint.pth'):
        path = ckpt_dir / name
        if path.exists():
            return path
    return None

def run_pretrain(train_dir: Path, val_dir: Path, output_root: Path, experiment: str, epochs: int, batch_size: int, val_batch_size: int, grad_accum_steps: int, num_workers: int, resume: Path | None = None, visualize_every: int = 10):
    output_root.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'train_ssl_freq_mae.py',
        '--dataset', 'imagenet',
        '--data_root', str(DATA_ROOT),
        '--train_dir', str(train_dir),
        '--val_dir', str(val_dir),
        '--output_dir', str(output_root),
        '--experiment', experiment,
        '--epochs', str(epochs),
        '--batch_size', str(batch_size),
        '--val_batch_size', str(val_batch_size),
        '--num_workers', str(num_workers),
        '--grad_accum_steps', str(grad_accum_steps),
        '--base_lr', '1.5e-4',
        '--warmup_epochs', '40' if epochs > 10 else '1',
        '--image_size', '224',
        '--block_size', '16',
        '--patch_size', '16',
        '--mask_ratio', str(MASK_RATIO),
        '--timm_model_name', TIMM_TINY_MODEL,
        '--timm_pretrained',
        '--decoder_emb_dim', str(DECODER_EMB_DIM),
        '--decoder_layer', str(DECODER_LAYERS),
        '--decoder_head', str(DECODER_HEADS),
        '--visualize_every', str(visualize_every),
        '--amp',
    ]

    if resume is not None and Path(resume).exists():
        cmd.extend(['--resume', str(resume)])

    print('Command:')
    print(' '.join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True, cwd=str(REPO_DIR))

smoke_train_dir, smoke_val_dir = build_smoke_subset(train_dir, val_dir, SMOKE_ROOT)
print('Smoke train dir:', smoke_train_dir)
print('Smoke val dir:', smoke_val_dir)

run_pretrain(
    train_dir=smoke_train_dir,
    val_dir=smoke_val_dir,
    output_root=RUNS_ROOT,
    experiment=SMOKE_EXPERIMENT,
    epochs=1,
    batch_size=8,
    val_batch_size=8,
    grad_accum_steps=1,
    num_workers=2,
    resume=None,
    visualize_every=1,
)

print('Smoke test finished. If this cell succeeded, you can start the full pretrain.')


## 正式训练

这一格会直接启动正式预训练。

- 训练数据使用完整 ImageNet
- checkpoint 自动保存在 `RUNS_ROOT/FULL_EXPERIMENT/`
- 如果之前已经训练过，会优先从 `last_checkpoint_norm_pix.pth` 自动续跑
- 如果显存不够，先把上面配置格里的 `TRAIN_BATCH_SIZE` 再降一档


In [ ]:
resume_ckpt = find_resume_ckpt(RUNS_ROOT, FULL_EXPERIMENT)
print('Resume checkpoint:', resume_ckpt if resume_ckpt is not None else 'none')

run_pretrain(
    train_dir=train_dir,
    val_dir=val_dir,
    output_root=RUNS_ROOT,
    experiment=FULL_EXPERIMENT,
    epochs=FULL_EPOCHS,
    batch_size=TRAIN_BATCH_SIZE,
    val_batch_size=VAL_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    resume=resume_ckpt,
    visualize_every=10,
)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/imagenet_ssl_runs


训练结束后，关键输出位置：

- 最优模型：`/content/drive/MyDrive/imagenet_ssl_runs/vit_tiny_ssl_imagenet_colab/best_freq-mae-ssl.pth`
- 最新断点：`/content/drive/MyDrive/imagenet_ssl_runs/vit_tiny_ssl_imagenet_colab/checkpoints/last_checkpoint_norm_pix.pth`
- TensorBoard 日志：`/content/drive/MyDrive/imagenet_ssl_runs/vit_tiny_ssl_imagenet_colab/`

如果你后面还要在这个仓库里继续做第二阶段训练，这个 `best_freq-mae-ssl.pth` 就是后续最常用的输入。